# 01 — Explore Data

Scratch notebook only — NOT part of the reproducible pipeline (the real
pipeline is the `.py` scripts run via `scripts/run_full_pipeline.sh`).

Use this to:
1. Sanity-check the NIH ChestX-ray14 data loads correctly and look at label distribution
2. Visually inspect a few real Hernia images before fine-tuning
3. After generation, visually compare real vs synthetic images side-by-side

**Run this from the project root** (clone the repo, `cd diffusion-medimg-project`, open this notebook).

## Setup — run once per session

In [ ]:
!pip install -q -r ../requirements.txt

import sys
sys.path.append("..")  # so `from src...` imports work from inside notebooks/

from src.utils.seed import set_seed, load_config
config = load_config("../configs/config.yaml")
set_seed(config["project"]["seed"])

## 1. Load NIH ChestX-ray14 and check label distribution

In [ ]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset(config["data"]["hf_dataset_id"], split="train")
print(f"Total images: {len(ds)}")
print(ds[0].keys())

In [ ]:
# Count how many images contain the target pathology — confirms the
# ~0.2% severe imbalance we expect for Hernia before committing GPU time
from src.data.prepare_data import filter_by_pathology

pathology = config["data"]["target_pathology"]
n_positive = sum(1 for ex in ds.select(range(5000)) if filter_by_pathology(ex, pathology))
print(f"In a 5000-image sample: {n_positive} contain '{pathology}' "
      f"({100*n_positive/5000:.2f}%)")

## 2. Visually inspect a few real target-pathology images

In [ ]:
import matplotlib.pyplot as plt

positive_examples = [ex for ex in ds.select(range(5000)) if filter_by_pathology(ex, pathology)][:6]

fig, axes = plt.subplots(1, len(positive_examples), figsize=(18, 3))
for ax, ex in zip(axes, positive_examples):
    ax.imshow(ex["image"], cmap="gray")
    ax.set_title(pathology)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. (Run AFTER Stage 1 of the pipeline) Inspect the exported fine-tuning set

In [ ]:
import os
from PIL import Image

export_dir = f"../{config['data']['image_export_dir']}"
files = sorted(f for f in os.listdir(export_dir) if f.endswith(".png"))[:6]

fig, axes = plt.subplots(1, len(files), figsize=(18, 3))
for ax, f in zip(axes, files):
    img = Image.open(os.path.join(export_dir, f))
    ax.imshow(img)
    ax.set_title(f)
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Total fine-tuning images exported: {len(os.listdir(export_dir)) // 2}")  # /2 since each has a .txt caption

## 4. (Run AFTER Stage 3 of the pipeline) Compare real vs synthetic side-by-side

This is a quick visual sanity check only — the actual memorization/
diversity evidence comes from `src/evaluation/diversity_check.py` (Stage 4),
not from eyeballing a few images here.

In [ ]:
synth_dir = f"../{config['generation']['output_dir']}"
synth_files = sorted(f for f in os.listdir(synth_dir) if f.endswith(".png"))[:6]

fig, axes = plt.subplots(2, 6, figsize=(20, 6))
for ax, f in zip(axes[0], files):
    ax.imshow(Image.open(os.path.join(export_dir, f)))
    ax.set_title("REAL", fontsize=10)
    ax.axis("off")
for ax, f in zip(axes[1], synth_files):
    ax.imshow(Image.open(os.path.join(synth_dir, f)))
    ax.set_title("SYNTHETIC", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. (Run AFTER Stage 5) Pull classifier results from W&B for a quick look

Full results live on your W&B dashboard — this just pulls the summary
table for a quick inline check without leaving the notebook.

In [ ]:
import wandb
import pandas as pd

api = wandb.Api()
runs = api.runs(f"{config['project']['wandb_project']}", filters={"jobType": "classifier-training"})
latest_run = runs[0]
table = latest_run.summary.get("classifier_results_by_ratio")
print("Open the W&B dashboard for the full interactive table:", latest_run.url)